# 01 · FX vendor: OANDA vs yfinance

Validate OANDA v20 mid bars vs Yahoo FX for the 7 v1 pairs.

**Window:** last ~728 calendar days (Yahoo 1H wall). Skip live fetches when
`OANDA_API_TOKEN` is missing or placeholder.


## 0. Imports & Config


In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.ingestion.credentials_env import read_credential
from data.ingestion.fx_fetcher import (
    G10_V1_PAIRS,
    NY_CLOSE_HOUR,
    NY_CLOSE_TZ,
    fetch_fx_ohlcv,
)

token = read_credential("OANDA_API_TOKEN") or ""
HAS_OANDA = bool(token) and token.lower() not in {"keyhere", "changeme", "your_token_here", ""}
print("HAS_OANDA=", HAS_OANDA)
print("pairs=", G10_V1_PAIRS)


## 1. Timing convention table


In [ ]:
timing = pd.DataFrame([
    {"vendor": "OANDA v20", "bar": "D", "timestamp_meaning": "candle start (aligned)",
     "params": f"alignmentTimezone={NY_CLOSE_TZ}, dailyAlignment={NY_CLOSE_HOUR}",
     "vs_ny_1700": "target day boundary"},
    {"vendor": "OANDA v20", "bar": "H1 / M1", "timestamp_meaning": "candle open time (UTC)",
     "params": "granularity H1/M1", "vs_ny_1700": "intraday; resample to Yahoo stamps"},
    {"vendor": "yfinance", "bar": "1d / 1h", "timestamp_meaning": "bar end (Yahoo FX)",
     "params": "EURUSD=X etc.", "vs_ny_1700": "not guaranteed NY 17:00 — compare overlaps only"},
])
display(timing)


## 2. Per-pair MAE / MAPE / return corr (skip if no keys)


In [ ]:
end = pd.Timestamp.utcnow().normalize()
start = end - pd.Timedelta(days=728)
rows = []

if not HAS_OANDA:
    print("SKIP live OANDA vs yfinance — set OANDA_API_TOKEN in config/credentials.env")
else:
    for pair in G10_V1_PAIRS:
        try:
            oanda = fetch_fx_ohlcv(pair, start, end, interval="1h", source="oanda")
            yf = fetch_fx_ohlcv(pair, start, end, interval="1h", source="yfinance")
            if oanda is None or yf is None or oanda.empty or yf.empty:
                rows.append({"pair": pair, "verdict": "fail", "reason": "empty"})
                continue
            o = oanda.set_index(pd.to_datetime(oanda["date"]))[["open", "high", "low", "close"]]
            y = yf.set_index(pd.to_datetime(yf["date"]))[["open", "high", "low", "close"]]
            both = o.join(y, lsuffix="_o", rsuffix="_y", how="inner")
            if both.empty:
                rows.append({"pair": pair, "verdict": "fail", "reason": "no overlap"})
                continue
            mae = (both["close_o"] - both["close_y"]).abs().mean()
            mape = (mae / both["close_y"].abs().mean()) if both["close_y"].abs().mean() else np.nan
            corr = both["close_o"].pct_change().corr(both["close_y"].pct_change())
            missing_yahoo = len(o.index.difference(y.index))
            rows.append({
                "pair": pair, "n_overlap": len(both), "mae_close": mae, "mape_close": mape,
                "ret_corr": corr, "oanda_bars_yahoo_lacks": missing_yahoo,
                "verdict": "pass" if corr and corr > 0.95 else "fail",
            })
        except Exception as exc:
            rows.append({"pair": pair, "verdict": "fail", "reason": repr(exc)})

verdict = pd.DataFrame(rows)
display(verdict)


## 3. Verdict


In [ ]:
if verdict.empty:
    print("No live comparison run.")
else:
    fails = verdict.loc[verdict.get("verdict", pd.Series(dtype=str)) == "fail"]
    print("failures:", len(fails))
    if not fails.empty:
        print("Flag in 05_strategies/s3_fx_trend/s3_algorithm_notes.md if persistent.")
        display(fails)
